
# 2.2 Assignment Statements and Target Semantics

## Overview

Assignment is one of Python's fundamental state-changing mechanisms.

Although the syntax of ordinary assignment appears simple, its execution semantics depend on expression evaluation, target structure, name binding, object protocols, and the order in which individual assignment targets are processed.

An assignment statement can bind a name, update an object attribute, replace a container element, modify a slice, or distribute values across multiple targets.

These operations have different consequences for object identity, shared mutable state, exception handling, and program correctness.

Advanced Python programming therefore requires an understanding of assignment as a language-level operation rather than merely as a mechanism for storing values.

In quantitative finance and data science, assignment semantics are especially important when updating portfolio holdings, maintaining trading ledgers, modifying financial datasets, and implementing stateful computational systems.

This notebook examines Python assignment statements through formal language semantics, runtime protocols, abstract syntax trees, numerical applications, and transactional state-management principles.

## Learning Objectives

By the end of this notebook, you will be able to:

1. Understand the formal structure of Python assignment statements.
2. Classify valid assignment targets and explain their behavior.
3. Analyze right-hand-side expression evaluation.
4. Explain the order in which multiple targets are assigned.
5. Distinguish name binding from attribute and item assignment.
6. Investigate attribute assignment through descriptors and properties.
7. Understand subscription and slice assignment protocols.
8. Analyze recursive and overlapping assignment targets.
9. Identify observable side effects during assignment.
10. Explain why multi-target assignment does not guarantee atomicity.
11. Diagnose failures that leave partially modified program state.
12. Inspect assignment statements using abstract syntax trees and bytecode.
13. Design controlled state transitions for financial applications.
14. Implement rollback-safe validation and staged updates.
15. Verify assignment behavior using deterministic tests and invariants.
16. Apply professional Python coding conventions to stateful analytical software.

## Technical Scope

This notebook focuses on assignment statements, assignment targets, evaluation order, and state changes.

Detailed iterable unpacking is covered in Section 2.4.

Augmented assignment and in-place special methods are covered in Section 2.5.

Variable annotations and assignment expressions are developed in Sections 2.6 and 2.7.

**Python Version:** 3.10 or later

**Dependencies:** Python Standard Library

**Difficulty:** Advanced / Expert

**Application Domain:** Quantitative Finance, Data Science, and Software Engineering



## 2.2.1 Assignment Grammar and Target Classification

### Definition

An assignment statement evaluates one or more expressions and assigns the resulting object or objects to one or more targets.

Python assignment is not restricted to variable names.

The nature of the assignment target determines how the resulting state change occurs.

### 1. Simple Name Targets

A name target establishes or changes a binding in the appropriate namespace.

Example:

    portfolio_value = 100000.0

The identifier becomes associated with the evaluated object.

### 2. Attribute Targets

An attribute target assigns a value to an attribute of an object.

Example:

    portfolio.cash = 5000.0

Attribute assignment may invoke the object's attribute-assignment machinery, including `__setattr__` and applicable descriptor protocols.

### 3. Subscription Targets

A subscription target assigns a value to an indexed or keyed location within an object.

Example:

    holdings["AAPL"] = 100

For ordinary user-defined objects, this operation is implemented through `__setitem__`.

### 4. Slice Targets

A slice target replaces a selected portion of a mutable sequence.

Example:

    prices[1:3] = [101.0, 102.0]

The assignment is handled by the target object's supported item-assignment behavior.

### 5. Target Lists

Assignment targets can be grouped into lists or tuples of targets.

Example:

    price, quantity = 125.0, 100

The assigned iterable must satisfy the target structure's unpacking requirements.

### 6. Starred Targets

A target list may contain a starred target that collects a variable number of items.

Example:

    first, *middle, last = observations

The starred target receives a list of the remaining items.

### 7. Assignment vs. Comparison

Assignment:

    x = 10

Equality comparison:

    x == 10

These expressions have fundamentally different purposes and grammatical roles.

### 8. Assignment vs. Assignment Expressions

Ordinary assignment uses `=` as part of an assignment statement.

Assignment expressions use `:=` within permitted expression contexts.

Their grammar and restrictions differ.

### 9. Assignment Target Validity

Not every Python expression can appear on the left-hand side of an assignment.

For example, a numeric literal or an arithmetic expression cannot ordinarily be used as an assignment target.

### 10. Assignment and Object State

Different targets produce different kinds of effects:

| Target | Principal Effect |
|---|---|
| Name | Bind or rebind an identifier |
| Attribute | Assign an object attribute |
| Subscription | Assign an indexed or keyed item |
| Slice | Replace a selected sequence region |
| Target list | Distribute values recursively |

### Core Principle

The right-hand side determines the value to assign.

The left-hand target structure determines how that value is incorporated into program state.


In [1]:

# ============================================================
# ASSIGNMENT TARGET CLASSIFICATION
# ============================================================

from dataclasses import dataclass


@dataclass
class PortfolioState:
    """Represent mutable portfolio state."""

    cash_usd: float
    holdings: dict[str, int]


portfolio = PortfolioState(
    cash_usd=100_000.0,
    holdings={
        "AAPL": 50,
        "MSFT": 30,
    },
)


# 1. Simple name target
portfolio_nav_usd = 125_000.0


# 2. Attribute target
portfolio.cash_usd = 95_000.0


# 3. Subscription target
portfolio.holdings["AAPL"] = 75


# 4. Slice target
closing_prices = [
    100.0,
    101.0,
    102.0,
    103.0,
    104.0,
]

closing_prices[1:4] = [
    101.5,
    102.5,
    103.5,
]


# 5. Target-list assignment
ticker, quantity = "JPM", 40


# 6. Starred target
first_price, *intermediate_prices, last_price = (
    closing_prices
)


print("ASSIGNMENT TARGET REPORT")
print("-" * 60)

print(f"Portfolio NAV: ${portfolio_nav_usd:,.2f}")
print(f"Cash Balance: ${portfolio.cash_usd:,.2f}")

print("AAPL Quantity:", portfolio.holdings["AAPL"])
print("Closing Prices:", closing_prices)

print("Unpacked Ticker:", ticker)
print("Unpacked Quantity:", quantity)

print("First Price:", first_price)
print("Intermediate Prices:", intermediate_prices)
print("Last Price:", last_price)


ASSIGNMENT TARGET REPORT
------------------------------------------------------------
Portfolio NAV: $125,000.00
Cash Balance: $95,000.00
AAPL Quantity: 75
Closing Prices: [100.0, 101.5, 102.5, 103.5, 104.0]
Unpacked Ticker: JPM
Unpacked Quantity: 40
First Price: 100.0
Intermediate Prices: [101.5, 102.5, 103.5]
Last Price: 104.0



### **Analytical Interpretation**

**1. Simple Name Assignment**

The statement assigning `portfolio_nav_usd` establishes a binding between a name and a floating-point object.

It does not modify the `PortfolioState` instance.

**2. Attribute Assignment**

The statement `portfolio.cash_usd = 95_000.0` updates an attribute of the existing portfolio object.

The portfolio object's identity remains unchanged.

**3. Subscription Assignment**

The statement `portfolio.holdings["AAPL"] = 75` modifies the dictionary referenced by the portfolio's `holdings` attribute.

It does not create a new portfolio object.

**4. Slice Assignment**

The slice assignment replaces selected observations within the existing list.

The list retains its identity, although its contents change.

**5. Target-List Assignment**

The two-target assignment distributes the evaluated values across the corresponding target names.

**6. Starred Target Assignment**

The starred target collects the intermediate observations into a new list.

**7. Target-Dependent Semantics**

Assignment may produce name rebinding, attribute modification, container mutation, or recursive distribution of values.

These operations are not interchangeable.

**8. Financial Data Implications**

When updating financial data, the exact assignment target determines which state is affected and which other references may observe the change.

**9. Engineering Principle**

Understanding the category of each assignment target is essential for predicting the side effects of state-changing Python statements.



## 2.2.2 Right-Hand-Side Evaluation and Assignment Order

### 1. General Evaluation Rule

In ordinary assignment, Python evaluates the right-hand-side expression before assigning the resulting value to the left-hand-side target.

Conceptually:

    target = expression

follows two stages:

1. Evaluate the expression.
2. Assign the resulting object to the target.

### 2. Expression Evaluation

Python evaluates expressions according to its language-defined evaluation-order rules.

When multiple expressions appear in an expression list, their evaluation proceeds from left to right.

### 3. Target Evaluation

Complex assignment targets may require additional evaluation.

For example:

    container[key] = value

requires the container and key expressions to be evaluated during target assignment.

### 4. Multiple Target Assignment

When an assignment statement contains multiple target lists, Python assigns to them from left to right.

### 5. Evaluation vs. Assignment

Evaluation of the right-hand side and application of the assignment target are distinct operations.

A side effect may occur during either stage.

### 6. Chained Assignment

For:

    a = b = expression

the expression is evaluated once, and the resulting object is assigned to each target from left to right.

### 7. Recursive Target Assignment

When targets are grouped into a tuple or list pattern, Python distributes the evaluated iterable's items according to the target structure.

### 8. Side Effects

The evaluation of an expression may:

- Read mutable state.
- Execute a function.
- Raise an exception.
- Modify external objects.

Complex target expressions may also produce observable side effects.

### 9. Financial Implications

Evaluation order matters when:

- Computing transaction values.
- Updating positions.
- Maintaining portfolio ledgers.
- Calling stateful calculation functions.
- Assigning to dynamically selected containers.

### Core Principle

For ordinary assignment, the right-hand-side value is determined before the assignment target is applied.

The assignment target may then perform additional state-changing operations.


In [2]:

# ============================================================
# ASSIGNMENT EVALUATION ORDER
# ============================================================

execution_trace = []


class TrackedContainer:
    """Record item-assignment events."""

    def __init__(self) -> None:
        self.data = {}

    def __setitem__(self, key: str, value: float) -> None:
        execution_trace.append(
            f"setitem({key}, {value})"
        )

        self.data[key] = value


container = TrackedContainer()


def evaluate_rhs() -> float:
    """Evaluate the assignment value."""
    execution_trace.append("evaluate_rhs")
    return 125.0


def evaluate_container() -> TrackedContainer:
    """Evaluate the subscription's primary expression."""
    execution_trace.append("evaluate_container")
    return container


def evaluate_key() -> str:
    """Evaluate the subscription key."""
    execution_trace.append("evaluate_key")
    return "AAPL"


# The RHS is evaluated before the assignment target.
evaluate_container()[evaluate_key()] = evaluate_rhs()


print("EXECUTION ORDER")
print("-" * 60)

for index, event in enumerate(
    execution_trace,
    start=1,
):
    print(f"{index}. {event}")


expected_trace = [
    "evaluate_rhs",
    "evaluate_container",
    "evaluate_key",
    "setitem(AAPL, 125.0)",
]

assert execution_trace == expected_trace

assert container.data["AAPL"] == 125.0


EXECUTION ORDER
------------------------------------------------------------
1. evaluate_rhs
2. evaluate_container
3. evaluate_key
4. setitem(AAPL, 125.0)



### **Analytical Interpretation**

**1. Right-Hand-Side Evaluation**

The function `evaluate_rhs()` executes before the assignment target expressions.

Its returned value becomes the object to be assigned.

**2. Container Evaluation**

After the right-hand side is evaluated, Python evaluates the container expression involved in the subscription target.

**3. Key Evaluation**

The key expression is evaluated after the container expression.

**4. Item Assignment**

The resulting container and key are used to perform item assignment through `__setitem__`.

**5. Observable Execution Order**

The trace records the sequence in which the different operations occur.

It demonstrates that the written left-to-right appearance of the assignment statement does not imply that its target expressions execute before the right-hand side.

**6. Side-Effect Implications**

If either the right-hand-side expression or the target expressions produce side effects, those effects occur in the language-defined evaluation order.

**7. Quantitative Finance Implications**

Stateful financial functions may return different values depending on when portfolio data is read or modified.

Correct evaluation-order reasoning is therefore essential when implementing execution systems and portfolio updates.

**8. Engineering Principle**

Avoid hidden side effects in assignment expressions whenever practical.

For complex financial state transitions, compute and validate values explicitly before applying mutations.



## 2.2.3 Name Targets and Namespace Binding

### 1. Name Assignment

When an assignment target is a simple identifier, Python binds the name to the object produced by the right-hand-side expression.

Example:

    portfolio_nav = 100000.0

The binding occurs in the namespace determined by the current execution context and applicable scope declarations.

### 2. Local Name Binding

Assignment to a name inside a function generally creates or updates a local binding unless the name is declared otherwise.

A binding operation can determine a name's local classification throughout the function body.

### 3. Global Name Binding

The `global` statement directs applicable assignments to a module-level namespace.

It does not copy the referenced object or automatically create an independent state.

### 4. Nonlocal Name Binding

The `nonlocal` statement allows assignments to a binding in an appropriate enclosing function scope.

Unlike `global`, it requires an existing binding in a non-global enclosing scope.

### 5. Rebinding vs. Mutation

Name rebinding changes an identifier's associated object.

Mutation changes the state of an existing object.

### 6. Name Deletion

The `del` statement can remove an existing name binding.

Deleting a name does not necessarily destroy its referenced object.

### 7. Namespace Lookup

A name's meaning depends on its scope and namespace.

Two different functions may use the same identifier while maintaining independent local bindings.

### 8. Assignment and Closures

Assignments involving enclosing variables can affect the behavior of closures that retain access to those bindings.

### 9. Global State Risks

Assignments to shared module-level state can introduce hidden dependencies between analytical components.

### 10. Notebook Execution State

Interactive notebooks preserve variables across cell executions.

The order in which cells are executed may therefore influence the current namespace contents.

### Engineering Principle

Prefer explicit state ownership and function interfaces over unnecessary mutation of global variables.


In [3]:

# ============================================================
# NONLOCAL ASSIGNMENT AND VERSIONED STATE
# ============================================================

from math import isfinite


def create_versioned_risk_policy(
    initial_limit: float,
):
    """
    Create a risk policy with controlled enclosing state.

    Returns
    -------
    tuple
        A risk-limit reader and a validated update function.
    """
    if not isfinite(initial_limit):
        raise ValueError("Initial limit must be finite.")

    if not 0 <= initial_limit <= 1:
        raise ValueError("Invalid initial limit.")

    current_limit = initial_limit
    version = 0

    def get_policy() -> tuple[float, int]:
        """Return the current limit and version."""
        return current_limit, version

    def update_policy(new_limit: float) -> None:
        """Validate and update the enclosing bindings."""
        nonlocal current_limit, version

        if (
            isinstance(new_limit, bool)
            or not isinstance(new_limit, (int, float))
            or not isfinite(new_limit)
            or not 0 <= new_limit <= 1
        ):
            raise ValueError("Invalid risk limit.")

        current_limit = float(new_limit)
        version += 1

    return get_policy, update_policy


get_policy, update_policy = create_versioned_risk_policy(
    0.25
)

print("INITIAL POLICY")
print(get_policy())

update_policy(0.30)

print("\nUPDATED POLICY")
print(get_policy())

assert get_policy() == (0.30, 1)


# ============================================================
# INDEPENDENT ENCLOSING STATES
# ============================================================

get_other_policy, update_other_policy = (
    create_versioned_risk_policy(0.15)
)

update_other_policy(0.20)

assert get_policy() == (0.30, 1)
assert get_other_policy() == (0.20, 1)

print("\nINDEPENDENT POLICY")
print(get_other_policy())


# ============================================================
# FAILURE PRESERVES THE EXISTING BINDINGS
# ============================================================

before_failure = get_policy()

try:
    update_policy(1.50)

except ValueError:
    pass

assert get_policy() == before_failure

print("\nPOLICY AFTER INVALID UPDATE")
print(get_policy())


INITIAL POLICY
(0.25, 0)

UPDATED POLICY
(0.3, 1)

INDEPENDENT POLICY
(0.2, 1)

POLICY AFTER INVALID UPDATE
(0.3, 1)



### **Analytical Interpretation**

**1. Enclosing Name Binding**

The variables `current_limit` and `version` belong to the enclosing function's execution environment.

The returned functions retain access to these bindings.

**2. Nonlocal Assignment**

The `nonlocal` declaration allows the update function to rebind names in its enclosing function scope.

Without the appropriate declaration, assignments would instead be treated as local to the nested function.

**3. Versioned State**

Each successful update changes the risk limit and increases the version counter.

The version provides a simple mechanism for tracking successful state changes.

**4. Independent Closures**

Separate invocations of the factory function construct independent enclosing environments.

Updating one policy does not change the other policy's bindings.

**5. Validation Before Assignment**

The invalid-update example validates the proposed value before changing either state variable.

Therefore, the expected validation failure leaves the previous policy unchanged.

**6. Multi-Statement Limitations**

The two updates inside the function remain separate Python assignment operations.

The implementation does not provide thread-safe or transactional guarantees for concurrent callers.

**7. Quantitative Finance Implications**

Configurable risk policies should expose explicit update operations and prevent invalid numerical parameters from being incorporated into analytical state.

**8. Engineering Principle**

Name-binding control is useful for maintaining state, but stronger consistency requirements may require locks, immutable state objects, or transactional update mechanisms.



## 2.2.4 Attribute Assignment and Descriptor Semantics

### 1. Attribute Assignment

Attribute assignment has the general form:

    object.attribute = value

The right-hand side is evaluated before the assignment target is applied.

### 2. Attribute Storage

Many ordinary Python objects store instance attributes in an instance dictionary.

However, attribute storage and assignment behavior are not universally implemented through a simple dictionary.

### 3. Attribute Assignment Protocol

For user-defined classes, attribute assignment is handled through Python's attribute-assignment mechanisms.

The special method:

    __setattr__(self, name, value)

can customize assignment behavior.

### 4. Object-Level Validation

An object may validate or transform a value before storing it.

For example, a risk policy may reject invalid portfolio concentration limits.

### 5. Descriptors

Descriptors are objects implementing selected attribute-access methods, including:

    __get__()
    __set__()
    __delete__()

A descriptor can control how an attribute is accessed or assigned.

### 6. Data Descriptors

A descriptor defining `__set__` or `__delete__` is classified as a data descriptor.

Data descriptors can take precedence over ordinary instance-dictionary attributes during attribute resolution.

### 7. Properties

The `property` mechanism provides a common way to define managed attributes.

A property setter can validate assignments.

### 8. Class vs. Instance Attributes

A value read through an instance may originate from a class attribute.

Assigning to that same attribute name on the instance may create an instance-specific attribute rather than changing the class attribute.

This behavior can differ when descriptors are involved.

### 9. Assignment Failure

Attribute assignment may raise an exception.

Therefore, assignment to an attribute should not be assumed to be a trivial storage operation.

### 10. Advanced Considerations

Attribute assignment can involve:

- Validation.
- Conversion.
- Logging.
- Descriptor dispatch.
- Custom attribute storage.
- Additional observable side effects.

### Quantitative Applications

Managed attributes are useful for:

- Risk-limit validation.
- Order configuration.
- Portfolio constraints.
- Financial model parameters.
- Controlled state transitions.

### Core Principle

An attribute assignment is an operation governed by the target object's behavior, not merely a direct change to a named memory location.


In [4]:

# ============================================================
# DATA DESCRIPTOR FOR FINANCIAL RISK LIMITS
# ============================================================

from math import isfinite


class BoundedWeight:
    """Validate access to a portfolio weight attribute."""

    def __set_name__(self, owner, name) -> None:
        self.storage_name = f"_{name}"

    def __get__(self, instance, owner):
        if instance is None:
            return self

        return instance.__dict__[self.storage_name]

    def __set__(self, instance, value) -> None:
        if (
            isinstance(value, bool)
            or not isinstance(value, (int, float))
            or not isfinite(value)
            or not 0 <= value <= 1
        ):
            raise ValueError(
                "Weight must be finite and between 0 and 1."
            )

        instance.__dict__[self.storage_name] = float(value)


class AllocationPolicy:
    """Store validated allocation limits."""

    max_position_weight = BoundedWeight()
    max_sector_weight = BoundedWeight()

    def __init__(
        self,
        max_position_weight: float,
        max_sector_weight: float,
    ) -> None:
        self.max_position_weight = max_position_weight
        self.max_sector_weight = max_sector_weight


policy = AllocationPolicy(
    max_position_weight=0.25,
    max_sector_weight=0.40,
)

print("INITIAL POLICY")
print(policy.max_position_weight)
print(policy.max_sector_weight)


# Descriptor-controlled assignment
policy.max_position_weight = 0.30

assert policy.max_position_weight == 0.30


# Invalid assignment is rejected by the descriptor.
try:
    policy.max_sector_weight = 1.50

except ValueError as error:
    print("\nREJECTED ASSIGNMENT")
    print(type(error).__name__)

assert policy.max_sector_weight == 0.40


# ============================================================
# CLASS ATTRIBUTE SHADOWING
# ============================================================

class RiskDefaults:
    max_weight = 0.25


first_policy = RiskDefaults()
second_policy = RiskDefaults()

first_policy.max_weight = 0.35

assert first_policy.max_weight == 0.35
assert second_policy.max_weight == 0.25
assert RiskDefaults.max_weight == 0.25

print("\nCLASS VS. INSTANCE ATTRIBUTES")
print("First Instance:", first_policy.max_weight)
print("Second Instance:", second_policy.max_weight)
print("Class Default:", RiskDefaults.max_weight)


INITIAL POLICY
0.25
0.4

REJECTED ASSIGNMENT
ValueError

CLASS VS. INSTANCE ATTRIBUTES
First Instance: 0.35
Second Instance: 0.25
Class Default: 0.25



### **Analytical Interpretation**

**1. Descriptor-Controlled Assignment**

The attributes `max_position_weight` and `max_sector_weight` are managed by instances of the `BoundedWeight` descriptor.

Their assignment behavior is not equivalent to unrestricted storage in an ordinary instance dictionary.

**2. Attribute Name Discovery**

The `__set_name__()` method records the attribute name used by the owning class.

This allows each descriptor to select a corresponding internal storage key.

**3. Attribute Reading**

The `__get__()` method retrieves the validated value stored in the instance.

**4. Attribute Writing**

The `__set__()` method executes when an assignment targets the managed attribute.

The descriptor validates the proposed value before updating storage.

**5. Validation Failure**

An assignment of an invalid weight raises `ValueError`.

Because the example validates before writing, the existing attribute value remains unchanged.

**6. Class Attribute Shadowing**

For the ordinary class attribute example, assigning `first_policy.max_weight` creates an instance-specific attribute.

The original class-level value remains available to other instances.

**7. Descriptor vs. Ordinary Attribute Behavior**

Descriptor-controlled assignment and ordinary instance-attribute shadowing follow different mechanisms.

This distinction affects how configuration objects behave when updated.

**8. Quantitative Finance Implications**

Descriptors and properties can centralize validation for financial parameters, reducing the risk of invalid allocation limits.

**9. Encapsulation Limitations**

The example provides validation through the intended attribute interface, but it is not a security boundary.

Other forms of direct object manipulation may bypass the demonstrated validation.

**10. Engineering Principle**

Attribute assignment semantics should be included in the design of financial data models whenever assigned values are subject to domain-specific constraints.



## 2.2.5 Subscription and Slice Assignment

### 1. Subscription Assignment

Subscription assignment has the general form:

    container[key] = value

The target object's supported item-assignment behavior determines the result.

### 2. The `__setitem__` Protocol

User-defined objects can support item assignment through:

    __setitem__(self, key, value)

The implementation may validate, transform, reject, or store the value.

### 3. Sequence Assignment

Mutable sequences may support integer-index assignment.

Example:

    prices[0] = 105.0

### 4. Mapping Assignment

Mutable mappings may support key-based assignment.

Example:

    positions["AAPL"] = 100

### 5. Slice Assignment

A slice assignment replaces a selected region of a mutable sequence.

Example:

    prices[1:3] = [101.0, 102.0]

For built-in lists, an ordinary slice assignment may change the number of elements.

### 6. Extended Slice Assignment

An extended slice uses a non-default step:

    prices[::2] = replacement_values

For built-in lists, the number of replacement elements must equal the number of selected positions when assigning to an extended slice with a non-unit step.

### 7. Slice Objects

Python represents slice specifications using `slice` objects.

For example:

    slice(1, 5, 2)

A custom container can inspect such objects in `__setitem__`.

### 8. Immutable Containers

Tuples and strings do not support ordinary item assignment.

Attempts to assign to their elements raise errors.

### 9. Read-Only Views

Some objects expose read-only interfaces even when their underlying data structures are mutable.

### 10. Side Effects and Validation

Subscription assignment may raise exceptions or produce side effects determined by the target's implementation.

### Quantitative Applications

Subscription assignment is common in:

- Historical price correction.
- Position quantity updates.
- Trading signal storage.
- Data preprocessing.
- Portfolio aggregation.
- Time-series window replacement.

### Core Principle

Subscription and slice assignment are operations performed by the target object and should be understood through its supported protocols.


In [5]:

# ============================================================
# SUBSCRIPTION AND SLICE ASSIGNMENT
# ============================================================

assignment_trace = []


class AuditedSequence:
    """Track index and slice assignment operations."""

    def __init__(self, values):
        self.values = list(values)

    def __setitem__(self, key, value) -> None:
        assignment_trace.append(
            (key, value)
        )

        self.values[key] = value

    def __getitem__(self, key):
        return self.values[key]


series = AuditedSequence(
    [100.0, 101.0, 102.0, 103.0, 104.0]
)


# Single-index assignment
series[0] = 99.0


# Ordinary slice replacement
series[1:3] = [101.5, 102.5]


# Extended slice replacement
series[::2] = [100.0, 103.0, 105.0]


print("UPDATED PRICE SERIES")
print(series.values)


print("\nASSIGNMENT AUDIT")
print("-" * 60)

for key, value in assignment_trace:
    print(f"Target: {key!r}, Value: {value!r}")


# ============================================================
# INVALID EXTENDED SLICE ASSIGNMENT
# ============================================================

try:
    series[::2] = [1.0, 2.0]

except ValueError as error:
    print("\nEXPECTED SLICE ERROR")
    print(type(error).__name__)


# ============================================================
# IMMUTABLE TARGET
# ============================================================

immutable_prices = (100.0, 101.0, 102.0)

try:
    immutable_prices[0] = 110.0

except TypeError:
    print("\nTuple item assignment rejected.")


UPDATED PRICE SERIES
[100.0, 101.5, 103.0, 103.0, 105.0]

ASSIGNMENT AUDIT
------------------------------------------------------------
Target: 0, Value: 99.0
Target: slice(1, 3, None), Value: [101.5, 102.5]
Target: slice(None, None, 2), Value: [100.0, 103.0, 105.0]

EXPECTED SLICE ERROR
ValueError

Tuple item assignment rejected.



### **Analytical Interpretation**

**1. Custom Item Assignment**

The `AuditedSequence` object implements `__setitem__()` to record target assignments before delegating storage to its internal list.

**2. Integer Indexing**

The first assignment replaces one selected element of the list.

**3. Ordinary Slice Assignment**

The second assignment replaces a contiguous region with a sequence of values.

**4. Extended Slice Assignment**

The third assignment selects elements at regular intervals.

The replacement sequence must match the number of selected positions for the demonstrated extended slice.

**5. Slice Objects**

The recorded assignment keys reveal that Python uses `slice` objects to represent slice targets.

**6. Assignment Errors**

The invalid extended slice assignment raises `ValueError` because the replacement length does not satisfy the built-in list's requirements.

**7. Immutable Containers**

A tuple does not support ordinary item assignment.

The attempted modification raises `TypeError`.

**8. Custom Protocol Behavior**

Custom containers can implement additional validation or audit logic in `__setitem__()`.

However, custom implementations may differ from built-in list behavior.

**9. Financial Data Implications**

Slice assignment can modify historical observations in place and may therefore affect other components referencing the same sequence.

**10. Engineering Principle**

Financial data-processing systems should distinguish between in-place data corrections and the creation of independent revised datasets.



## 2.2.6 Recursive Targets and Overlapping Assignments

### 1. Recursive Assignment Targets

Assignment targets may contain nested target lists.

Python distributes values according to the recursive structure of the target specification.

### 2. Unpacking Requirements

Without a starred target, the assigned iterable must contain the required number of elements.

The assignment may raise an exception when the shape is incompatible.

### 3. Starred Targets

A starred target collects a variable number of elements.

The complete rules for extended unpacking are examined in Section 2.4.

### 4. Multiple Assignment Targets

A single assignment statement may contain multiple assignment targets.

Example:

    a = b = expression

The expression is evaluated once, and the resulting object is assigned to each target from left to right.

### 5. Simultaneous-Looking Assignment

For:

    a, b = b, a

the right-hand-side values are evaluated before the target assignments.

This enables variable swapping.

### 6. Overlapping Targets

Assignment targets may interact when an earlier target assignment changes a name or object used by a later target.

### 7. Left-to-Right Target Assignment

Python applies targets from left to right.

This rule is particularly important when assignment targets overlap.

### 8. Financial Risks

Overlapping targets may create unintended state changes in:

- Position lists.
- Dynamic index calculations.
- Portfolio configuration.
- Mutable research containers.

### 9. Atomicity Distinction

Evaluating all right-hand-side values before assignment does not guarantee that all subsequent target assignments succeed together.

### Core Principle

The combination of right-hand-side evaluation and left-to-right target assignment can produce results that differ from a naive interpretation of assignment as a simultaneous update.


In [6]:

# ============================================================
# OVERLAPPING ASSIGNMENT TARGETS
# ============================================================

position_quantities = [0, 1]

index = 0

# The RHS is evaluated first.
# Then assignment targets are processed left to right.
index, position_quantities[index] = 1, 2

print("OVERLAPPING TARGETS")
print("-" * 60)

print("Updated Index:", index)
print("Position Quantities:", position_quantities)

assert index == 1
assert position_quantities == [0, 2]


# ============================================================
# REVERSE TARGET ORDER
# ============================================================

position_quantities = [0, 1]

index = 0

position_quantities[index], index = 2, 1

print("\nREVERSE TARGET ORDER")
print("-" * 60)

print("Updated Index:", index)
print("Position Quantities:", position_quantities)

assert index == 1
assert position_quantities == [2, 1]


# ============================================================
# NESTED TARGET ASSIGNMENT
# ============================================================

trade_record = (
    "AAPL",
    (100, 185.50),
)

ticker, (quantity, execution_price) = trade_record

assert ticker == "AAPL"
assert quantity == 100
assert execution_price == 185.50


# ============================================================
# CHAINED ASSIGNMENT AND SHARED IDENTITY
# ============================================================

first_positions = second_positions = []

assert first_positions is second_positions

first_positions.append("AAPL")

assert second_positions == ["AAPL"]


print("\nNESTED ASSIGNMENT")
print(ticker, quantity, execution_price)

print("\nCHAINED REFERENCE")
print(first_positions is second_positions)


OVERLAPPING TARGETS
------------------------------------------------------------
Updated Index: 1
Position Quantities: [0, 2]

REVERSE TARGET ORDER
------------------------------------------------------------
Updated Index: 1
Position Quantities: [2, 1]

NESTED ASSIGNMENT
AAPL 100 185.5

CHAINED REFERENCE
True



### **Analytical Interpretation**

**1. Overlapping Targets**

The first overlapping assignment updates `index` before assigning to the subscription target.

Consequently, the subscription uses the updated index.

**2. Reversed Assignment Order**

Reversing the target order changes which list element is modified.

This demonstrates that target assignment is not a simultaneous operation over all targets.

**3. Right-Hand-Side Evaluation**

The values on the right-hand side are determined before the target assignments begin.

The subsequently executed target assignments still follow left-to-right order.

**4. Recursive Target Assignment**

The nested target structure distributes the trade record's elements across the corresponding names.

**5. Chained Assignment**

Both `first_positions` and `second_positions` reference the same list object.

The expression constructing the list is evaluated once.

**6. Shared Mutable State**

Modifying the list through one name changes the contents visible through the other name.

**7. Financial System Risks**

Overlapping target assignments can modify unintended portfolio elements when index variables or other target-dependent references are updated earlier in the same statement.

**8. Assignment Atomicity**

Although some assignments appear simultaneous, their individual target updates are not generally guaranteed to succeed or fail as one indivisible operation.

**9. Engineering Principle**

Avoid overlapping assignment targets in critical financial state updates when their execution order is not immediately obvious.



## 2.2.7 Evaluation Side Effects and Assignment Tracing

### 1. Observable Side Effects

An assignment may produce observable effects beyond changing a simple name binding.

Potential effects include:

- Calling functions.
- Updating mutable objects.
- Executing property setters.
- Invoking custom assignment methods.
- Recording audit information.
- Raising exceptions.

### 2. Right-Hand-Side Side Effects

An expression on the right-hand side may perform operations before its result is assigned.

### 3. Target-Side Side Effects

Attribute and subscription targets may execute user-defined methods.

These methods may validate, transform, or record the assigned value.

### 4. Assignment Tracing

Execution traces record the order in which operations occur.

They can help investigate unexpected assignment behavior.

### 5. Attribute Assignment Tracing

The `__setattr__` method can intercept attribute assignments on instances of user-defined classes.

### 6. Subscription Assignment Tracing

The `__setitem__` method can intercept item-assignment operations.

### 7. Side-Effect Ordering

The observed side effects depend on the applicable evaluation and assignment rules.

### 8. State Consistency

An exception may occur after one or more side effects have already taken place.

### 9. Financial Applications

Assignment tracing can support:

- Portfolio state debugging.
- Order-processing diagnostics.
- Position update auditing.
- Risk-rule enforcement.
- Financial data integrity analysis.

### 10. Production Considerations

Logging and auditing should not introduce unexpected state modifications or obscure the underlying financial calculations.

### Core Principle

Assignment is an executable operation that may invoke user-defined behavior and produce externally observable state changes.


In [7]:

# ============================================================
# ASSIGNMENT SIDE-EFFECT TRACING
# ============================================================

events = []


class StateAudit:
    """Record attribute and item assignment operations."""

    def __init__(self) -> None:
        object.__setattr__(self, "_positions", {})

    def __setattr__(self, name, value) -> None:
        events.append(
            f"attribute:{name}={value}"
        )

        object.__setattr__(
            self,
            name,
            value,
        )

    def __setitem__(self, key, value) -> None:
        events.append(
            f"item:{key}={value}"
        )

        self._positions[key] = value


def produce_value(label, value):
    """Record the production of an assignment value."""
    events.append(f"rhs:{label}")
    return value


state = StateAudit()


# Attribute assignment
state.cash_usd = produce_value(
    "cash",
    50_000.0,
)


# Subscription assignment
state["AAPL"] = produce_value(
    "position",
    100,
)


print("ASSIGNMENT EXECUTION TRACE")
print("-" * 60)

for index, event in enumerate(events, start=1):
    print(f"{index}. {event}")


assert events == [
    "rhs:cash",
    "attribute:cash_usd=50000.0",
    "rhs:position",
    "item:AAPL=100",
]

assert state.cash_usd == 50_000.0
assert state._positions["AAPL"] == 100


ASSIGNMENT EXECUTION TRACE
------------------------------------------------------------
1. rhs:cash
2. attribute:cash_usd=50000.0
3. rhs:position
4. item:AAPL=100



### **Analytical Interpretation**

**1. Right-Hand-Side Execution**

The function `produce_value()` executes before the corresponding target assignment.

Its execution is recorded in the audit trace.

**2. Attribute Assignment**

Assigning `cash_usd` invokes the custom `__setattr__()` method.

This method records the operation before delegating the actual attribute storage.

**3. Subscription Assignment**

Assigning the AAPL position invokes the custom `__setitem__()` method.

The method records the operation and updates the internal positions dictionary.

**4. User-Defined Assignment Behavior**

The assignment targets determine which special methods participate in the operation.

**5. State Mutation**

The two assignments modify the existing `StateAudit` instance rather than replacing the object referenced by `state`.

**6. Execution Trace**

The trace identifies the order of value production and target modification.

**7. Quantitative Finance Implications**

Assignment auditing can help identify unexpected portfolio state changes and trace the order in which financial updates occur.

**8. Audit Limitations**

An in-memory event list does not provide durable audit logging, transaction isolation, or protection against process failure.

**9. Engineering Principle**

Stateful assignment operations should have predictable side effects and should expose sufficient information for testing and debugging.



## 2.2.8 Assignment Failures and Partial State Changes

### 1. Assignment Failure

An assignment may fail when an expression or target operation raises an exception.

Potential causes include:

- Invalid target values.
- Type incompatibilities.
- Descriptor validation errors.
- Invalid container indices.
- Unsupported item assignment.
- Custom object restrictions.

### 2. Right-Hand-Side Failure

If right-hand-side evaluation raises an exception before target assignment begins, those target assignments do not occur.

However, side effects performed during right-hand-side evaluation may already have occurred.

### 3. Target Assignment Failure

A target assignment may fail after earlier assignment targets have already been updated.

### 4. Multi-Target Assignment Is Not Transactional

Python does not provide a general guarantee that every target in a multi-target assignment is updated atomically.

### 5. Partial State Changes

A failure in a later target assignment may leave earlier changes visible.

### 6. Database Transaction Analogy

Database transactions commonly provide atomicity guarantees.

An ordinary Python assignment statement does not automatically provide equivalent transactional semantics.

### 7. Validation Before Mutation

One strategy is to validate proposed values before applying state changes.

This prevents certain predictable failures.

### 8. Staged State Updates

For more complex operations, changes can be applied to independent temporary state before being committed.

### 9. Rollback

A rollback strategy attempts to restore a previous valid state after failure.

However, restoration itself may fail or be incomplete when external side effects are involved.

### 10. Concurrency

Thread synchronization and assignment atomicity are separate concerns.

A lock can protect a defined critical section when all cooperating operations follow the same locking policy.

The presence of Python's Global Interpreter Lock in some implementations should not be treated as a substitute for explicit concurrency control.

### 11. Financial Applications

Partial updates can corrupt:

- Cash balances.
- Portfolio holdings.
- Trading ledgers.
- Risk exposure records.
- Transaction identifiers.
- Model state.

### Core Principle

Complex financial state transitions should be validated and staged before modifying authoritative state.


In [8]:

# ============================================================
# PARTIAL STATE UPDATE AFTER ASSIGNMENT FAILURE
# ============================================================

class GuardedPortfolio:
    """Demonstrate a validation failure during assignment."""

    def __init__(self) -> None:
        self.positions = {"AAPL": 100}
        self._max_weight = 0.25

    @property
    def max_weight(self) -> float:
        return self._max_weight

    @max_weight.setter
    def max_weight(self, value: float) -> None:
        if not 0 <= value <= 1:
            raise ValueError(
                "Weight must be between 0 and 1."
            )

        self._max_weight = value


portfolio = GuardedPortfolio()


# ============================================================
# MULTI-TARGET ASSIGNMENT
# ============================================================

try:
    (
        portfolio.positions["AAPL"],
        portfolio.max_weight,
    ) = (
        120,
        1.50,
    )

except ValueError as error:
    print("ASSIGNMENT FAILED")
    print(type(error).__name__)


# The first target was already updated.
assert portfolio.positions["AAPL"] == 120

# The second target rejected the invalid value.
assert portfolio.max_weight == 0.25


print("\nSTATE AFTER FAILURE")
print("-" * 55)

print("AAPL Shares:", portfolio.positions["AAPL"])
print("Maximum Weight:", portfolio.max_weight)


# ============================================================
# VALIDATION BEFORE APPLYING CHANGES
# ============================================================

proposed_quantity = 130
proposed_weight = 0.30

if (
    proposed_quantity > 0
    and 0 <= proposed_weight <= 1
):
    portfolio.positions["AAPL"] = proposed_quantity
    portfolio.max_weight = proposed_weight


assert portfolio.positions["AAPL"] == 130
assert portfolio.max_weight == 0.30

print("\nSTATE AFTER VALIDATED UPDATE")
print(portfolio.positions)
print(portfolio.max_weight)


ASSIGNMENT FAILED
ValueError

STATE AFTER FAILURE
-------------------------------------------------------
AAPL Shares: 120
Maximum Weight: 0.25

STATE AFTER VALIDATED UPDATE
{'AAPL': 130}
0.3



### **Analytical Interpretation**

**1. Multi-Target Evaluation**

The right-hand-side values are evaluated before the assignment targets begin processing.

**2. Left-to-Right Target Execution**

The position quantity is assigned before the risk-limit attribute.

**3. Partial Update**

The position quantity changes successfully.

The subsequent attribute assignment raises an exception because the proposed weight is invalid.

**4. State After Failure**

The portfolio remains partially modified despite the failure of the complete statement.

**5. Validation Before Mutation**

The second approach validates the proposed values before changing either attribute.

This prevents the demonstrated invalid-input failure.

**6. Validation Limitations**

Prevalidation alone does not guarantee that every subsequent assignment will succeed.

Unexpected exceptions, concurrent changes, or custom setters may still cause partial updates.

**7. Financial Consistency**

A trading system must avoid applying a position change while failing to apply an associated cash, risk, or transaction-record update.

**8. Transactional Design**

A stronger design stages modifications in temporary state and commits them only after the complete operation has passed validation.

**9. Engineering Principle**

Multi-target Python syntax should not be mistaken for an all-or-nothing financial transaction.



## 2.2.9 Assignment Semantics in AST and Bytecode

### 1. Abstract Syntax Trees

Python's `ast` module provides a structural representation of source code.

Assignment statements are represented by specific node categories.

### 2. Assignment Nodes

Important AST node types include:

| Node | Meaning |
|---|---|
| `Assign` | Ordinary assignment |
| `AnnAssign` | Annotated assignment |
| `AugAssign` | Augmented assignment |
| `NamedExpr` | Assignment expression |
| `Name` | Identifier reference or target |
| `Attribute` | Attribute access or target |
| `Subscript` | Indexed or sliced access or target |
| `Tuple` | Tuple expression or target structure |
| `List` | List expression or target structure |
| `Starred` | Starred target or expression |

### 3. Assignment Targets and Context

AST nodes can carry context information such as:

- `Load`
- `Store`
- `Del`

A complex target may contain nested nodes with different contexts.

### 4. Source-Code Inspection

AST analysis can identify where names, attributes, and subscriptions appear as assignment targets.

### 5. Bytecode

In CPython, source code is compiled into bytecode instructions.

Relevant instruction families may include operations for:

- Storing names.
- Storing attributes.
- Storing subscription values.
- Loading and evaluating expressions.

### 6. Bytecode Version Dependence

CPython bytecode can change between versions.

Instruction names and sequences should not be assumed to remain fixed.

### 7. AST vs. Runtime Behavior

AST nodes describe program structure.

They do not determine whether custom attribute setters or item-assignment methods will succeed during execution.

### 8. Static Analysis Applications

Assignment inspection can support:

- Code auditing.
- Mutation-risk identification.
- Variable-binding analysis.
- Detection of selected code smells.
- Review of financial state modifications.

### 9. Static Analysis Limitations

Static inspection cannot generally establish full transactional safety or prove the absence of runtime side effects.

### Core Principle

AST and bytecode inspection help explain how Python represents assignment, but runtime testing is necessary to verify actual state changes.


In [9]:

import ast
import dis


# ============================================================
# ASSIGNMENT TARGET STRUCTURE
# ============================================================

source_code = """
portfolio_nav = 100000.0
portfolio.cash_usd = 50000.0
positions["AAPL"] = 75
prices[1:3] = [101.0, 102.0]
first, second = 10, 20
"""


tree = ast.parse(source_code)

print("ASSIGNMENT TARGET ANALYSIS")
print("-" * 70)

for statement in tree.body:

    if isinstance(statement, ast.Assign):

        for target in statement.targets:
            print(
                f"Line {statement.lineno:<3}"
                f"Target: {type(target).__name__:<12}"
                f"Source: {ast.unparse(target)}"
            )


# ============================================================
# NAME CONTEXT ANALYSIS
# ============================================================

print("\nNAME CONTEXTS")
print("-" * 70)

for node in ast.walk(tree):

    if isinstance(node, ast.Name):
        print(
            f"{node.id:<20}"
            f"{type(node.ctx).__name__}"
        )


# ============================================================
# CPYTHON BYTECODE INSPECTION
# ============================================================

def update_portfolio(
    portfolio,
    positions,
    cash_value,
    shares,
):
    """Perform attribute and item assignments."""
    portfolio.cash_usd = cash_value
    positions["AAPL"] = shares


print("\nSTORE-RELATED BYTECODE INSTRUCTIONS")
print("-" * 70)

for instruction in dis.get_instructions(
    update_portfolio
):
    if instruction.opname.startswith("STORE"):
        print(
            f"{instruction.opname:<25}"
            f"{instruction.argrepr}"
        )


ASSIGNMENT TARGET ANALYSIS
----------------------------------------------------------------------
Line 2  Target: Name        Source: portfolio_nav
Line 3  Target: Attribute   Source: portfolio.cash_usd
Line 4  Target: Subscript   Source: positions['AAPL']
Line 5  Target: Subscript   Source: prices[1:3]
Line 6  Target: Tuple       Source: (first, second)

NAME CONTEXTS
----------------------------------------------------------------------
portfolio_nav       Store
portfolio           Load
positions           Load
prices              Load
first               Store
second              Store

STORE-RELATED BYTECODE INSTRUCTIONS
----------------------------------------------------------------------
STORE_ATTR               cash_usd
STORE_SUBSCR             



### **Analytical Interpretation**

**1. Assignment Statement Representation**

Ordinary assignment statements are represented by `ast.Assign` nodes.

Each node contains information about its assignment targets and right-hand-side expression.

**2. Target Classification**

The example identifies name, attribute, subscription, slice, and tuple-target structures.

**3. Nested Target Contexts**

A name used as the base of an attribute or subscription assignment may appear in a `Load` context even when the overall target is being assigned.

**4. Structural Inspection**

The AST representation allows assignment targets to be analyzed without executing the source code.

**5. Bytecode Inspection**

The disassembly demonstrates how CPython represents assignment operations using lower-level instructions.

**6. Implementation Dependence**

The exact bytecode representation depends on the Python implementation and version.

**7. Static Analysis Limitations**

Identifying an attribute assignment does not establish whether the target uses a property, descriptor, or custom attribute-assignment method.

**8. Runtime Verification**

Runtime tests are necessary to determine whether target assignments succeed and which objects are modified.

**9. Quantitative Finance Implications**

Static analysis can help identify potentially state-changing operations in trading, data-processing, and portfolio-management code.

**10. Engineering Principle**

Structural analysis and runtime behavioral testing should be used together when auditing financial state modifications.



## 2.2.10 Common Pitfalls and State Integrity

### 1. Assuming Assignment Is Atomic

Multiple assignment targets are not guaranteed to update as one indivisible transaction.

### 2. Ignoring Target Evaluation Order

Overlapping targets may produce unexpected results when earlier assignments affect later target expressions.

### 3. Assuming Attribute Assignment Is Simple Storage

Properties, descriptors, and custom assignment methods can introduce validation and side effects.

### 4. Assuming Subscription Assignment Always Succeeds

Custom containers and built-in sequences may reject invalid keys, indices, or replacement values.

### 5. Confusing Name Assignment with Mutation

Rebinding a name is different from changing the state of an existing object.

### 6. Ignoring Shared Mutable References

Item or attribute assignment may modify an object accessed by multiple parts of a program.

### 7. Ignoring Partial State Updates

An exception may occur after earlier targets have already been modified.

### 8. Misunderstanding Augmented Assignment

Augmented assignment may perform in-place modification and then attempt to assign the result back to the target.

This can produce surprising behavior when the final target assignment is invalid.

Detailed augmented assignment semantics are covered in Section 2.5.

### 9. Assuming Prevalidation Guarantees Atomicity

Validation reduces predictable failures but does not automatically provide transactional behavior.

### 10. Uncontrolled Attribute Setters

Setters may modify multiple related fields.

Failure during a setter may leave partially modified object state.

### 11. Concurrency Assumptions

Multiple state-changing operations require an explicit concurrency model when accessed from multiple execution contexts.

### 12. Hidden Notebook State

Interactive execution may preserve partially modified objects after an exception.

### Prevention Strategies

- Understand the assignment target category.
- Make evaluation order explicit when necessary.
- Avoid overlapping targets in critical updates.
- Validate numerical inputs before mutation.
- Separate proposed state from authoritative state.
- Stage complex modifications.
- Protect shared state with appropriate synchronization.
- Verify failure-path behavior.
- Test rollback and input preservation.
- Keep analytical state transitions auditable.

### Core Principle

Reliable state management requires understanding assignment semantics and designing explicit consistency boundaries.


In [10]:

# ============================================================
# AUGMENTED ASSIGNMENT PREVIEW
# ============================================================

# This behavior will be studied in detail in Section 2.5.

immutable_outer = (
    [1, 2],
)

nested_list = immutable_outer[0]

try:
    immutable_outer[0] += [3]

except TypeError:
    print("Assignment back to tuple rejected.")


# The list was modified before the final assignment failed.
assert immutable_outer[0] is nested_list
assert immutable_outer[0] == [1, 2, 3]


print("\nSTATE AFTER FAILED AUGMENTED ASSIGNMENT")
print(immutable_outer)


Assignment back to tuple rejected.

STATE AFTER FAILED AUGMENTED ASSIGNMENT
([1, 2, 3],)



### **Analytical Interpretation**

**1. Augmented Assignment Evaluation**

The expression attempts to augment the list referenced by the tuple's first element.

**2. In-Place List Mutation**

The list's in-place addition modifies its contents.

**3. Assignment Back to the Target**

Python subsequently attempts to assign the result back to the tuple element.

**4. Immutable Target Failure**

The tuple does not support item assignment.

Therefore, the final assignment step raises `TypeError`.

**5. Mutation Despite Failure**

The referenced list has already been modified.

The exception does not automatically reverse that mutation.

**6. Atomicity Implications**

This example demonstrates that a failed assignment operation may still have observable side effects.

**7. Quantitative Finance Risks**

Similar issues may arise when state-changing operations modify mutable data before later validation or assignment fails.

**8. Engineering Principle**

Do not assume that catching an assignment-related exception restores the previous program state.

Verify actual object state after failures and design explicit transactional safeguards when consistency is required.



## 2.2.11 Practical Application: Transactional Portfolio State Transition Engine

### Objective

Develop a portfolio transaction-processing engine that demonstrates advanced Python assignment semantics while maintaining consistency between cash balances, security positions, and realized trading profits.

The engine must distinguish between ordinary assignment operations and application-level transactional state updates.

### Research Problem

A portfolio ledger maintains several related state variables:

- Available cash.
- Security positions.
- Average acquisition costs.
- Realized trading P&L.
- Processed transaction identifiers.
- Ledger version.

An individual trade may require multiple state changes.

If one operation succeeds and another fails, the ledger may become inconsistent unless the updates are carefully coordinated.

### System Architecture

    Incoming Trade Fills
             |
             v
       Input Validation
             |
             v
       Acquire State Lock
             |
             v
       Copy Current State
             |
             v
       Stage Trade Updates
             |
             v
       Validate Constraints
             |
       +-----+-----+
       |           |
       v           v
    Failure      Success
       |           |
       v           v
    Discard      Commit
    Staged       New State
    Changes        |
                   v
             Version Update
                   |
                   v
             Ledger Snapshot

### Functional Requirements

1. Represent trade fills using immutable records.
2. Support BUY and SELL transactions.
3. Validate prices, quantities, commissions, and identifiers.
4. Maintain long-only security positions.
5. Calculate average acquisition costs.
6. Calculate realized P&L on security sales.
7. Update available cash after transactions.
8. Reject transactions exceeding available cash.
9. Reject sales exceeding available holdings.
10. Detect duplicate transaction identifiers.
11. Process multiple trades as one logical batch.
12. Stage all modifications before committing them.
13. Preserve ledger state when batch validation fails.
14. Produce independent immutable state snapshots.
15. Maintain a ledger version counter.
16. Verify financial accounting invariants.

### Financial Definitions

**BUY Transaction**

    CashOutflow =
        Quantity * ExecutionPrice + Commission

    NewShares =
        PreviousShares + Quantity

    NewAverageCost =
        (PreviousShares * PreviousAverageCost + CashOutflow)
        / NewShares

**SELL Transaction**

    CashInflow =
        Quantity * ExecutionPrice - Commission

    RealizedPnL =
        Quantity * (ExecutionPrice - AverageCost)
        - Commission

    RemainingShares =
        PreviousShares - Quantity

### Financial Assumptions

- All holdings are long-only.
- Quantities are positive integers.
- All cash flows are denominated in USD.
- Short selling is prohibited.
- Negative cash balances are prohibited.
- Transaction fees are included in acquisition cost for purchases.
- Selling fees are deducted from realized P&L.
- The average-cost accounting method is used.
- No financing costs, taxes, dividends, or corporate actions are modeled.
- Historical or current market prices are not used for unrealized P&L.
- The model uses binary floating-point arithmetic.

### Transactional Consistency

The engine performs updates on temporary state.

If a trade fails validation, the temporary state is discarded without changing the authoritative ledger.

Successful batches are committed while holding a synchronization lock.

### Important Limitations

The implementation provides application-level consistency for cooperating operations within one process.

It is not a durable database transaction system and does not provide distributed transaction guarantees or crash-safe recovery.

### Technical Requirements

- Python 3.10 or later.
- Python standard library only.
- Explicit type annotations.
- Immutable trade and position records.
- Independent staged state.
- Controlled attribute assignment.
- Subscription-based position updates.
- Consistent locking.
- Deterministic calculations.
- Automated rollback and financial invariant tests.


In [11]:

# ============================================================
# TRANSACTIONAL PORTFOLIO STATE TRANSITION ENGINE
# ============================================================

from collections.abc import Sequence
from dataclasses import dataclass
from math import isfinite
from threading import RLock
from typing import Literal


Side = Literal["BUY", "SELL"]


def _finite_number(value: object) -> bool:
    """Return whether a value is a finite real number."""
    return (
        isinstance(value, (int, float))
        and not isinstance(value, bool)
        and isfinite(value)
    )


@dataclass(frozen=True, slots=True)
class TradeFill:
    """Represent a validated trade execution."""

    fill_id: str
    ticker: str
    side: Side
    quantity: int
    price_usd: float
    fee_usd: float

    def __post_init__(self) -> None:
        """Validate the execution record."""

        if (
            not isinstance(self.fill_id, str)
            or not self.fill_id.strip()
            or not isinstance(self.ticker, str)
            or not self.ticker.strip()
        ):
            raise ValueError(
                "Fill ID and ticker must be nonempty strings."
            )

        if self.side not in ("BUY", "SELL"):
            raise ValueError(
                "Trade side must be BUY or SELL."
            )

        if (
            isinstance(self.quantity, bool)
            or not isinstance(self.quantity, int)
            or self.quantity <= 0
        ):
            raise ValueError(
                "Quantity must be a positive integer."
            )

        if (
            not _finite_number(self.price_usd)
            or self.price_usd <= 0
        ):
            raise ValueError(
                "Price must be finite and positive."
            )

        if (
            not _finite_number(self.fee_usd)
            or self.fee_usd < 0
        ):
            raise ValueError(
                "Fee must be finite and nonnegative."
            )


@dataclass(frozen=True, slots=True)
class Position:
    """Store a security quantity and average cost."""

    shares: int
    average_cost_usd: float


@dataclass(frozen=True, slots=True)
class LedgerSnapshot:
    """
    Store an independent portfolio ledger snapshot.

    Positions are represented as an immutable tuple
    of ticker-position pairs.
    """

    version: int
    cash_usd: float
    realized_pnl_usd: float
    positions: tuple[tuple[str, Position], ...]


print("Portfolio ledger data models initialized.")


Portfolio ledger data models initialized.



### **Analytical Interpretation**

**1. Immutable Trade Records**

The `TradeFill` dataclass is frozen to prevent ordinary modification of an execution record after initialization.

**2. Assignment-Time Validation**

The `__post_init__()` method validates the fields assigned during object construction.

Invalid records are rejected before they can be used by the portfolio engine.

**3. Financial Type Contracts**

Trade quantities must be positive integers, while prices and fees must be finite numerical values.

Boolean values are explicitly excluded from numeric inputs.

**4. Position Representation**

The `Position` class stores the security quantity and average acquisition cost as one logical record.

**5. Independent Snapshots**

Ledger snapshots store immutable position records in tuples, reducing the risk of accidental modification through shared mutable containers.

**6. State Separation**

Trade records, position records, and ledger state have separate responsibilities.

This makes the eventual assignment operations easier to reason about and test.

**7. Numerical Limitations**

The code uses binary floating-point values for monetary amounts.

Production-grade financial accounting may require decimal arithmetic, currency-specific precision, and explicit rounding conventions.

**8. Engineering Principle**

Validation and immutable data models make assignment behavior more predictable, but transactional guarantees must still be implemented at the state-management level.


In [12]:

class PortfolioLedger:
    """
    Maintain a long-only portfolio ledger.

    Transactions are staged in independent temporary
    state before being committed under a lock.

    The ledger uses average-cost accounting.
    """

    def __init__(
        self,
        initial_cash_usd: float,
    ) -> None:
        """Initialize a validated portfolio ledger."""

        if (
            not _finite_number(initial_cash_usd)
            or initial_cash_usd < 0
        ):
            raise ValueError(
                "Initial cash must be finite and nonnegative."
            )

        self._cash_usd = float(initial_cash_usd)
        self._realized_pnl_usd = 0.0
        self._positions: dict[str, Position] = {}
        self._processed_fill_ids: set[str] = set()
        self._version = 0
        self._lock = RLock()

    def snapshot(self) -> LedgerSnapshot:
        """
        Return an independent immutable snapshot.

        The snapshot is constructed while holding
        the ledger's synchronization lock.
        """
        with self._lock:
            return LedgerSnapshot(
                version=self._version,
                cash_usd=self._cash_usd,
                realized_pnl_usd=self._realized_pnl_usd,
                positions=tuple(
                    sorted(self._positions.items())
                ),
            )

    def apply_batch(
        self,
        fills: Sequence[TradeFill],
    ) -> LedgerSnapshot:
        """
        Apply a batch of validated portfolio transactions.

        All financial updates are staged before the
        authoritative ledger state is changed.

        Raises
        ------
        ValueError
            If the batch is invalid, duplicates an
            existing fill, exceeds available cash,
            or attempts to sell unavailable shares.
        TypeError
            If a record is not a TradeFill instance.
        OverflowError
            If financial calculations become nonfinite.

        Notes
        -----
        Consistency depends on all participating
        operations using the same synchronization
        policy. This is not durable transaction storage.
        """
        if not fills:
            raise ValueError(
                "Transaction batch cannot be empty."
            )

        with self._lock:

            # =================================================
            # STAGE 1 — CREATE TEMPORARY STATE
            # =================================================

            staged_cash = self._cash_usd

            staged_realized_pnl = (
                self._realized_pnl_usd
            )

            staged_positions = (
                self._positions.copy()
            )

            staged_ids = (
                self._processed_fill_ids.copy()
            )

            # =================================================
            # STAGE 2 — PROCESS TRANSACTIONS
            # =================================================

            for fill in fills:

                if not isinstance(fill, TradeFill):
                    raise TypeError(
                        "Every fill must be a TradeFill."
                    )

                if fill.fill_id in staged_ids:
                    raise ValueError(
                        f"Duplicate fill ID: {fill.fill_id}"
                    )

                staged_ids.add(fill.fill_id)

                existing = staged_positions.get(
                    fill.ticker
                )

                previous_shares = (
                    existing.shares
                    if existing is not None
                    else 0
                )

                previous_basis = (
                    existing.average_cost_usd
                    if existing is not None
                    else 0.0
                )

                # =============================================
                # BUY TRANSACTION
                # =============================================

                if fill.side == "BUY":

                    outlay = (
                        fill.quantity * fill.price_usd
                        + fill.fee_usd
                    )

                    if outlay > staged_cash:
                        raise ValueError(
                            f"Insufficient cash: {fill.fill_id}"
                        )

                    new_shares = (
                        previous_shares
                        + fill.quantity
                    )

                    new_average_cost = (
                        previous_shares * previous_basis
                        + outlay
                    ) / new_shares

                    staged_cash -= outlay

                    staged_positions[fill.ticker] = Position(
                        shares=new_shares,
                        average_cost_usd=new_average_cost,
                    )

                # =============================================
                # SELL TRANSACTION
                # =============================================

                else:

                    if fill.quantity > previous_shares:
                        raise ValueError(
                            f"Insufficient shares: {fill.fill_id}"
                        )

                    proceeds = (
                        fill.quantity * fill.price_usd
                        - fill.fee_usd
                    )

                    realized_change = (
                        (
                            fill.price_usd
                            - previous_basis
                        )
                        * fill.quantity
                        - fill.fee_usd
                    )

                    staged_cash += proceeds

                    staged_realized_pnl += realized_change

                    remaining_shares = (
                        previous_shares
                        - fill.quantity
                    )

                    if remaining_shares > 0:

                        staged_positions[fill.ticker] = Position(
                            shares=remaining_shares,
                            average_cost_usd=previous_basis,
                        )

                    else:
                        del staged_positions[fill.ticker]

                # =============================================
                # STAGE 3 — VERIFY STAGED NUMERICAL STATE
                # =============================================

                staged_values = (
                    staged_cash,
                    staged_realized_pnl,
                    *(
                        position.average_cost_usd
                        for position in staged_positions.values()
                    ),
                )

                if not all(
                    isfinite(value)
                    for value in staged_values
                ):
                    raise OverflowError(
                        "Nonfinite staged financial state."
                    )

            # =================================================
            # STAGE 4 — COMMIT AUTHORITATIVE STATE
            # =================================================

            # These are sequential Python assignments.
            # Their consistency is protected for callers
            # following the ledger's locking policy.

            self._cash_usd = staged_cash

            self._realized_pnl_usd = staged_realized_pnl

            self._positions = staged_positions

            self._processed_fill_ids = staged_ids

            self._version += 1

            return self.snapshot()


print("Transactional portfolio ledger initialized.")


Transactional portfolio ledger initialized.



### **Analytical Interpretation**

**1. Assignment and Authoritative State**

The ledger maintains several related attributes representing the current financial state.

These attributes must remain consistent with one another.

**2. Staged State Construction**

Before processing transactions, the engine creates temporary copies of the current state.

Assignments during transaction processing modify the staged state rather than the authoritative ledger.

**3. Subscription Assignment**

Position updates use dictionary subscription assignment:

    staged_positions[ticker] = position

This changes the temporary dictionary.

**4. Attribute Assignment**

The authoritative ledger state is updated through attribute assignments after successful validation.

**5. Immutable Position Records**

Each position is represented by an immutable record.

This reduces the risk of staged and authoritative dictionaries sharing mutable position objects.

**6. BUY Accounting**

The engine reduces staged cash and calculates a new average acquisition cost.

Purchase commissions are included in the position's acquisition cost.

**7. SELL Accounting**

The engine increases cash by net sale proceeds and calculates realized P&L using the stored average cost.

**8. Validation Before Commit**

Invalid trades raise exceptions while the engine is still operating on temporary state.

Consequently, the normal validation-failure paths preserve the previous authoritative state.

**9. Commit Semantics**

The authoritative attributes are assigned sequentially under the synchronization lock.

The assignments are not inherently atomic at the Python language level.

**10. Synchronization**

The `RLock` allows cooperating ledger methods to read and update state without observing intermediate changes from one another.

This does not prevent direct external access to the ledger's internal attributes or provide crash-safe persistence.

**11. Snapshot Isolation**

The snapshot method returns immutable records representing the ledger's current state.

Subsequent ledger updates do not modify previously returned snapshots.

**12. Quantitative Finance Implications**

This design reduces the risk of a failed batch leaving cash, holdings, or realized P&L inconsistent.

**13. Engineering Principle**

For multi-object financial updates, use staged state and clearly defined commit boundaries rather than assuming that ordinary Python assignment statements provide transactional guarantees.


In [13]:

# ============================================================
# PORTFOLIO TRANSACTION SCENARIO
# ============================================================

ledger = PortfolioLedger(
    initial_cash_usd=100_000.0
)

fills = [
    TradeFill(
        fill_id="B1",
        ticker="AAPL",
        side="BUY",
        quantity=100,
        price_usd=180.0,
        fee_usd=5.0,
    ),
    TradeFill(
        fill_id="B2",
        ticker="MSFT",
        side="BUY",
        quantity=50,
        price_usd=400.0,
        fee_usd=5.0,
    ),
    TradeFill(
        fill_id="S1",
        ticker="AAPL",
        side="SELL",
        quantity=40,
        price_usd=190.0,
        fee_usd=4.0,
    ),
    TradeFill(
        fill_id="B3",
        ticker="AAPL",
        side="BUY",
        quantity=20,
        price_usd=200.0,
        fee_usd=2.0,
    ),
]


# ============================================================
# EXECUTE TRANSACTION BATCH
# ============================================================

initial_snapshot = ledger.snapshot()

final_snapshot = ledger.apply_batch(
    fills
)


# ============================================================
# GENERATE PORTFOLIO LEDGER REPORT
# ============================================================

print("=" * 70)
print("TRANSACTIONAL PORTFOLIO LEDGER REPORT")
print("=" * 70)

print(f"Ledger Version: {final_snapshot.version}")

print(
    f"Cash Balance: "
    f"${final_snapshot.cash_usd:,.2f}"
)

print(
    f"Realized P&L: "
    f"${final_snapshot.realized_pnl_usd:,.2f}"
)

print("\nOPEN POSITIONS")
print("-" * 70)

for ticker, position in final_snapshot.positions:

    print(
        f"{ticker:<10}"
        f"Shares: {position.shares:>6}  "
        f"Average Cost: ${position.average_cost_usd:>10,.4f}"
    )


# ============================================================
# TEST FAILED BATCH ROLLBACK
# ============================================================

before_failure = ledger.snapshot()

invalid_batch = [
    TradeFill(
        "B4", "GOOGL", "BUY",
        10, 100.0, 1.0,
    ),
    TradeFill(
        "S2", "AAPL", "SELL",
        500, 190.0, 3.0,
    ),
]

try:
    ledger.apply_batch(invalid_batch)

except ValueError as error:
    print("\nFAILED BATCH")
    print(error)


after_failure = ledger.snapshot()

assert before_failure == after_failure


print("\nSTATE INTEGRITY")
print("-" * 70)

print(
    "State Preserved After Failure:",
    before_failure == after_failure,
)

print(
    "Original Snapshot Preserved:",
    initial_snapshot.cash_usd == 100_000.0,
)


TRANSACTIONAL PORTFOLIO LEDGER REPORT
Ledger Version: 1
Cash Balance: $65,584.00
Realized P&L: $394.00

OPEN POSITIONS
----------------------------------------------------------------------
AAPL      Shares:     80  Average Cost: $  185.0625
MSFT      Shares:     50  Average Cost: $  400.1000

FAILED BATCH
Insufficient shares: S2

STATE INTEGRITY
----------------------------------------------------------------------
State Preserved After Failure: True
Original Snapshot Preserved: True



### Expected Results

#### Final Ledger State

| Metric | Expected Result |
|---|---:|
| Initial Cash | &#36;100,000.00 |
| Final Cash | &#36;65,584.00 |
| Realized P&L | +&#36;394.00 |
| Ledger Version | 1 |
| Open Positions | 2 |
| Failed Batch Committed? | No |
| State Preserved After Failure? | Yes |

#### Open Positions

| Ticker | Remaining Shares | Average Cost |
|---|---:|---:|
| AAPL | 80 | &#36;185.0625 |
| MSFT | 50 | &#36;400.1000 |

#### AAPL Transaction Analysis

**Initial Purchase**

100 shares at &#36;180 per share plus a &#36;5 commission.

Total acquisition cost:

    $18,005

Average acquisition cost:

    $180.05 per share

**Partial Sale**

40 shares sold at &#36;190 per share with a &#36;4 commission.

Realized P&L:

    40 * (190 - 180.05) - 4 = $394

**Additional Purchase**

20 shares purchased at &#36;200 per share with a &#36;2 commission.

Updated average cost:

    (60 * 180.05 + 20 * 200 + 2) / 80

    = $185.0625 per share

#### Financial Accounting Identity

Under the specified average-cost accounting conventions and in the absence of external cash flows:

    Cash + Remaining Acquisition Cost
        = Initial Cash + Realized P&L

For the final portfolio:

    $65,584 + $34,810
        = $100,000 + $394

    $100,394 = $100,394

This identity provides an additional consistency check for the ledger.

### Failure-Path Interpretation

The invalid batch first stages a valid purchase, then attempts to sell more AAPL shares than the portfolio owns.

The second transaction fails validation.

Because the changes were applied only to temporary state, the authoritative ledger remains unchanged.

The ledger version also remains unchanged.



### **Analytical Interpretation**

**1. Transaction Sequence**

The engine processes purchases and sales sequentially within a staged transaction batch.

Each trade updates temporary cash, positions, and realized P&L according to the specified accounting rules.

**2. Average-Cost Accounting**

The initial AAPL purchase establishes an acquisition cost that includes the purchase commission.

The subsequent sale realizes a gain relative to this average acquisition cost.

**3. Subsequent Position Update**

The later AAPL purchase increases the remaining share quantity and produces a new weighted average acquisition cost.

**4. Financial State Consistency**

Cash and remaining position acquisition costs satisfy the expected accounting identity.

This provides a useful numerical invariant for testing the implementation.

**5. Failed Batch**

The invalid transaction batch contains an attempted sale exceeding available holdings.

The engine rejects the batch.

**6. Staged Mutation**

Although the first transaction in the failed batch would have modified temporary state, those changes are discarded.

**7. Authoritative State Preservation**

The ledger snapshot before the failed batch equals the snapshot after the failure.

This demonstrates the intended rollback behavior for this validation failure.

**8. Assignment Semantics**

The implementation distinguishes ordinary state-changing assignments from an application-level commit procedure.

The commit consists of sequential assignments protected by a synchronization policy.

**9. Version Control**

The ledger version increases only after a successful batch completes.

Failed batches do not increase the version.

**10. Snapshot Independence**

Previously returned immutable snapshots remain unchanged after successful or unsuccessful ledger operations.

**11. Quantitative Finance Implications**

Transactional state management is important when executing trading strategies that require coordinated updates to cash balances, security quantities, cost bases, and realized P&L.

**12. Model Limitations**

The implementation does not calculate unrealized P&L, portfolio NAV, margin requirements, or market exposure.

Its transaction consistency is defined within a single process and does not constitute database-level durability or distributed transaction safety.

**13. Engineering Principle**

Reliable financial systems should explicitly distinguish computational results, staged state changes, committed state, and externally observable financial records.



## 2.2.12 Automated Testing and Invariants

### Testing Objective

Verify that portfolio state transitions preserve financial accounting relationships and that rejected transaction batches do not modify authoritative ledger state.

### Core Invariants

**Invariant 1 — Nonnegative Cash**

    Cash >= 0

The long-only portfolio must not borrow cash.

**Invariant 2 — Nonnegative Positions**

    Shares_i >= 0

Security quantities cannot become negative.

**Invariant 3 — Average Cost**

After a purchase:

    NewAverageCost =
        (PreviousCostBasis + PurchaseCost)
        / NewShareQuantity

**Invariant 4 — Realized P&L**

After a sale:

    RealizedPnLChange =
        Quantity * (SalePrice - AverageCost)
        - SaleCommission

**Invariant 5 — Accounting Consistency**

With no external cash flows:

    Cash + RemainingAcquisitionCost
        = InitialCash + CumulativeRealizedPnL

**Invariant 6 — Failed Batch Preservation**

If a transaction batch fails validation:

    StateAfter == StateBefore

**Invariant 7 — Transaction Uniqueness**

A transaction identifier must not be processed more than once.

**Invariant 8 — Version Consistency**

A successful batch increments the ledger version exactly once.

A rejected batch does not change the version.

**Invariant 9 — Snapshot Independence**

Previously returned snapshots must not change after subsequent ledger updates.

**Invariant 10 — Input Validation**

Invalid quantities, prices, commissions, and trade directions must be rejected.

### Testing Categories

- Reference calculation tests.
- Position quantity tests.
- Average-cost calculations.
- Realized P&L verification.
- Accounting identity tests.
- Transaction rollback tests.
- Duplicate identifier tests.
- Insufficient cash tests.
- Invalid input tests.
- Snapshot integrity tests.
- Version consistency tests.

### Testing Limitations

The test suite evaluates selected deterministic behavior.

It does not establish complete reliability under process crashes, hardware failures, concurrent external state access, or distributed execution.

### Core Principle

Financial state transitions should be verified using both application-level behavioral tests and independently derived accounting invariants.


In [14]:

import unittest

from math import isclose


class PortfolioLedgerTests(unittest.TestCase):
    """Test transactional portfolio state transitions."""

    def setUp(self) -> None:
        """Create a fresh ledger and trade batch."""

        self.ledger = PortfolioLedger(
            initial_cash_usd=100_000.0
        )

        self.fills = [
            TradeFill(
                "B1", "AAPL", "BUY",
                100, 180.0, 5.0,
            ),
            TradeFill(
                "B2", "MSFT", "BUY",
                50, 400.0, 5.0,
            ),
            TradeFill(
                "S1", "AAPL", "SELL",
                40, 190.0, 4.0,
            ),
            TradeFill(
                "B3", "AAPL", "BUY",
                20, 200.0, 2.0,
            ),
        ]

    def test_reference_cash(self) -> None:
        """Verify the final cash balance."""

        result = self.ledger.apply_batch(
            self.fills
        )

        self.assertAlmostEqual(
            result.cash_usd,
            65_584.0,
        )

    def test_reference_realized_pnl(self) -> None:
        """Verify realized trading profit."""

        result = self.ledger.apply_batch(
            self.fills
        )

        self.assertAlmostEqual(
            result.realized_pnl_usd,
            394.0,
        )

    def test_average_acquisition_cost(self) -> None:
        """Verify AAPL position quantity and average cost."""

        result = self.ledger.apply_batch(
            self.fills
        )

        positions = dict(result.positions)

        self.assertEqual(
            positions["AAPL"].shares,
            80,
        )

        self.assertAlmostEqual(
            positions["AAPL"].average_cost_usd,
            185.0625,
        )

    def test_accounting_identity(self) -> None:
        """Verify cash and acquisition-cost consistency."""

        result = self.ledger.apply_batch(
            self.fills
        )

        remaining_cost_basis = sum(
            position.shares
            * position.average_cost_usd
            for _, position in result.positions
        )

        self.assertTrue(
            isclose(
                result.cash_usd + remaining_cost_basis,
                100_000.0 + result.realized_pnl_usd,
                rel_tol=0.0,
                abs_tol=1e-8,
            )
        )

    def test_failed_batch_rollback(self) -> None:
        """Verify state preservation after a later trade fails."""

        self.ledger.apply_batch(self.fills)

        before = self.ledger.snapshot()

        invalid_batch = [
            TradeFill(
                "B4", "GOOGL", "BUY",
                10, 100.0, 1.0,
            ),
            TradeFill(
                "S2", "AAPL", "SELL",
                500, 190.0, 3.0,
            ),
        ]

        with self.assertRaises(ValueError):
            self.ledger.apply_batch(invalid_batch)

        self.assertEqual(
            self.ledger.snapshot(),
            before,
        )

    def test_duplicate_ids_within_batch(self) -> None:
        """Reject duplicate transaction identifiers."""

        with self.assertRaises(ValueError):
            self.ledger.apply_batch(
                [
                    self.fills[0],
                    self.fills[0],
                ]
            )

        self.assertEqual(
            self.ledger.snapshot().version,
            0,
        )

    def test_duplicate_existing_fill(self) -> None:
        """Prevent reprocessing an already committed fill."""

        self.ledger.apply_batch(self.fills)

        before = self.ledger.snapshot()

        with self.assertRaises(ValueError):
            self.ledger.apply_batch(
                [self.fills[0]]
            )

        self.assertEqual(
            self.ledger.snapshot(),
            before,
        )

    def test_insufficient_cash(self) -> None:
        """Reject purchases exceeding available cash."""

        invalid_fill = TradeFill(
            "X1", "AAPL", "BUY",
            1_000, 1_000.0, 0.0,
        )

        with self.assertRaises(ValueError):
            self.ledger.apply_batch(
                [invalid_fill]
            )

        self.assertEqual(
            self.ledger.snapshot().version,
            0,
        )

    def test_invalid_quantity(self) -> None:
        """Reject Boolean quantities."""

        with self.assertRaises(ValueError):
            TradeFill(
                "X1", "AAPL", "BUY",
                True, 180.0, 0.0,
            )

    def test_invalid_price(self) -> None:
        """Reject nonfinite execution prices."""

        with self.assertRaises(ValueError):
            TradeFill(
                "X1", "AAPL", "BUY",
                10, float("nan"), 0.0,
            )

    def test_empty_batch(self) -> None:
        """Reject an empty transaction batch."""

        with self.assertRaises(ValueError):
            self.ledger.apply_batch([])

    def test_snapshot_stability(self) -> None:
        """Verify that previous snapshots remain unchanged."""

        before = self.ledger.snapshot()

        self.ledger.apply_batch(self.fills)

        self.assertEqual(
            before.cash_usd,
            100_000.0,
        )

        self.assertEqual(
            before.positions,
            (),
        )

        self.assertEqual(
            before.version,
            0,
        )

    def test_successful_version_increment(self) -> None:
        """Verify one version increment per committed batch."""

        result = self.ledger.apply_batch(
            self.fills
        )

        self.assertEqual(
            result.version,
            1,
        )


# ============================================================
# EXECUTE THE TEST SUITE
# ============================================================

suite = unittest.defaultTestLoader.loadTestsFromTestCase(
    PortfolioLedgerTests
)

test_result = unittest.TextTestRunner(
    verbosity=2
).run(suite)

assert test_result.wasSuccessful()


test_accounting_identity (__main__.PortfolioLedgerTests.test_accounting_identity)
Verify cash and acquisition-cost consistency. ... ok
test_average_acquisition_cost (__main__.PortfolioLedgerTests.test_average_acquisition_cost)
Verify AAPL position quantity and average cost. ... ok
test_duplicate_existing_fill (__main__.PortfolioLedgerTests.test_duplicate_existing_fill)
Prevent reprocessing an already committed fill. ... ok
test_duplicate_ids_within_batch (__main__.PortfolioLedgerTests.test_duplicate_ids_within_batch)
Reject duplicate transaction identifiers. ... ok
test_empty_batch (__main__.PortfolioLedgerTests.test_empty_batch)
Reject an empty transaction batch. ... ok
test_failed_batch_rollback (__main__.PortfolioLedgerTests.test_failed_batch_rollback)
Verify state preservation after a later trade fails. ... ok
test_insufficient_cash (__main__.PortfolioLedgerTests.test_insufficient_cash)
Reject purchases exceeding available cash. ... ok
test_invalid_price (__main__.PortfolioLedgerTe


### **Analytical Interpretation**

**1. Cash Balance Verification**

The reference calculation confirms that the final cash balance reflects the combined cash flows from purchases, sales, and commissions.

**2. Realized P&L Verification**

The realized P&L test validates the average-cost accounting calculation for the partially closed AAPL position.

**3. Average Acquisition Cost**

The position test confirms that the remaining AAPL quantity and average acquisition cost are updated consistently.

**4. Accounting Identity**

The accounting test verifies that remaining acquisition cost plus cash equals initial cash plus cumulative realized P&L under the stated assumptions.

**5. Rollback Verification**

The failed-batch test confirms that a later invalid trade does not cause earlier staged changes to be committed.

**6. Duplicate Transaction Detection**

The tests verify that duplicate identifiers are rejected both within one batch and across previously committed batches.

**7. Insufficient Cash Protection**

The ledger rejects purchases that require more cash than is available.

**8. Runtime Input Validation**

The validation tests reject invalid quantities and nonfinite execution prices.

**9. Snapshot Independence**

The snapshot test confirms that a previously captured state remains unchanged after the ledger processes subsequent transactions.

**10. Version Consistency**

Successful batches increment the ledger version exactly once.

Rejected batches preserve the previous version.

**11. Assignment Semantics**

The tests demonstrate that a carefully designed state-management system can use ordinary Python assignments while providing stronger application-level consistency properties.

**12. Limitations**

Passing these tests does not establish crash safety, database durability, or full thread safety against arbitrary direct access to internal attributes.

**13. Quantitative Finance Implications**

State-integrity and accounting-invariant tests help identify financial software errors that might otherwise produce plausible but incorrect portfolio records.

**14. Engineering Principle**

Testing must address both successful state transitions and the behavior of the system when an intermediate operation fails.



## 2.2.13 Final Analytical Interpretation

### 1. Assignment as a Language-Level Operation

Python assignment is not limited to associating names with objects.

It may also invoke attribute setters, item-assignment protocols, slice replacement, and recursive target distribution.

### 2. Right-Hand-Side Evaluation

In ordinary assignment, Python evaluates the right-hand-side expression before applying the assignment target.

The evaluation itself may have side effects.

### 3. Target-Dependent Behavior

The assignment target determines how the evaluated object becomes part of program state.

Different target categories can invoke different runtime mechanisms.

### 4. Name Binding

Simple name assignment establishes or changes a binding in the appropriate namespace.

This operation differs from modifying an existing object's internal state.

### 5. Attribute Assignment

Attribute assignment may invoke custom methods, properties, and descriptors.

Such operations can validate values, modify state, or raise exceptions.

### 6. Subscription Assignment

Subscription targets use object-specific item-assignment behavior.

Mutable containers may support indexed or keyed updates, while immutable containers generally reject such assignments.

### 7. Recursive Targets

Target lists distribute values according to their nested structure.

The language applies target assignments in a defined order.

### 8. Overlapping Targets

Assignments can interact when earlier target updates influence later target expressions.

This can produce unexpected results even when the statement is syntactically valid.

### 9. Assignment Side Effects

Assignment operations may execute user-defined code and produce observable effects beyond basic storage.

### 10. Exception Semantics

An exception does not automatically restore the previous program state.

A failed assignment may leave earlier side effects visible.

### 11. Atomicity

Multi-target assignment does not generally guarantee all-or-nothing state updates.

Transactional consistency must be designed at the application level when required.

### 12. Static Analysis

Abstract syntax trees and bytecode inspection can reveal the structural form of assignment operations.

However, runtime behavior may depend on object-specific protocols and state.

### 13. Financial State Management

Portfolio systems often require coordinated changes to cash, positions, cost bases, and transaction records.

Applying these updates without a clear consistency policy can create invalid financial states.

### 14. Staged Updates

The practical application demonstrates how temporary state can be used to validate a complete transaction batch before updating authoritative records.

### 15. Rollback and State Preservation

Failure-path tests confirm that rejected batches preserve the previous ledger state under the implementation's assumptions.

### 16. Numerical Invariants

Financial accounting identities provide independent checks on the consistency of transaction processing.

### 17. Software Engineering Implications

Reliable financial systems should integrate:

- Explicit assignment semantics.
- Predictable side effects.
- Validated state transitions.
- Independent temporary state.
- Controlled commit procedures.
- Synchronization.
- Immutable snapshots.
- Accounting invariants.
- Automated rollback testing.

### 18. Limitations

The example does not implement persistent transaction storage, crash recovery, distributed coordination, or accounting-grade decimal precision.

These capabilities require additional design beyond Python's ordinary assignment semantics.

### Final Assessment

Advanced understanding of assignment requires reasoning about expression evaluation, target selection, object-specific protocols, and the order in which state changes occur.

The primary engineering objective is not merely to assign values correctly, but to ensure that related state modifications preserve the invariants required by the application.

For quantitative finance and data science, this distinction is essential when building reliable portfolio ledgers, data pipelines, and stateful analytical systems.



## Advanced Practice Exercises

### Exercise 1: Assignment Evaluation Order Analyzer

**Objective**

Develop a diagnostic tool that investigates the execution order of complex Python assignment statements.

**Tasks**

1. Construct expressions containing function calls on the right-hand side.
2. Construct attribute and subscription assignment targets.
3. Record right-hand-side evaluation events.
4. Record target-expression evaluation events.
5. Record assignment-method invocations.
6. Analyze chained assignment.
7. Analyze overlapping assignment targets.
8. Compare execution traces against predicted behavior.
9. Identify situations with observable side effects.
10. Produce an assignment execution report.

**Requirements**

- Standard Python only.
- Deterministic event tracing.
- Explicit expected execution sequences.
- No reliance on implementation-specific bytecode order.
- Comprehensive assertions.

---

### Exercise 2: Advanced Attribute Assignment Framework

**Objective**

Implement a managed financial configuration system using descriptors and custom attribute-assignment mechanisms.

**Tasks**

1. Define custom data descriptors.
2. Implement `__get__` and `__set__`.
3. Use `__set_name__` to manage internal storage.
4. Validate numerical risk parameters.
5. Reject invalid values.
6. Demonstrate class and instance attribute behavior.
7. Implement assignment auditing.
8. Investigate descriptor precedence.
9. Test assignment failure behavior.
10. Document encapsulation limitations.

**Requirements**

- Support multiple independent policy instances.
- Reject Boolean and nonfinite numerical values.
- Preserve valid values after failed validation.
- Include automated tests.
- Explain descriptor dispatch.

---

### Exercise 3: Subscription and Slice Assignment Engine

**Objective**

Develop a custom mutable financial time-series container.

**Tasks**

1. Implement item access and assignment.
2. Support integer indices.
3. Support slice assignment.
4. Validate replacement values.
5. Record assignment events.
6. Reject invalid slice lengths where appropriate.
7. Preserve chronological data consistency.
8. Implement independent data snapshots.
9. Test failed assignment behavior.
10. Compare the implementation with built-in list semantics.

**Requirements**

- Use `__getitem__` and `__setitem__`.
- Support slice objects.
- Define an explicit mutation policy.
- Implement informative exceptions.
- Include numerical data-integrity tests.

---

### Exercise 4: Assignment Failure and State Recovery

**Objective**

Investigate partial state modifications caused by assignment failures.

**Tasks**

1. Create a multi-target assignment with a failing later target.
2. Record state before assignment.
3. Execute the assignment.
4. Identify successful modifications before failure.
5. Implement validation before mutation.
6. Demonstrate why prevalidation alone is insufficient.
7. Implement a staged state-update mechanism.
8. Verify that rejected staged updates preserve authoritative state.
9. Introduce unexpected exceptions during processing.
10. Analyze the limitations of rollback.

**Requirements**

- Include attribute and subscription targets.
- Include at least five failure cases.
- Use deterministic tests.
- Document any operations that cannot be safely reversed.
- Explain the difference between rollback and transactional atomicity.

---

### Exercise 5: AST-Based Assignment Auditor

**Objective**

Develop a static analysis tool for identifying potentially state-changing assignment operations.

**Tasks**

1. Parse source code using `ast`.
2. Identify ordinary assignment statements.
3. Identify annotated assignments.
4. Identify augmented assignments.
5. Classify name targets.
6. Classify attribute targets.
7. Classify subscription and slice targets.
8. Detect nested target structures.
9. Identify potentially overlapping targets.
10. Generate a source-location report.
11. Calculate assignment complexity metrics.
12. Explain limitations of static analysis.

**Requirements**

- Do not execute analyzed source code.
- Python standard library only.
- Support nested functions and classes.
- Include source locations.
- Avoid claiming that static inspection proves runtime mutation behavior.

---

### Exercise 6: Transactional Portfolio Ledger Extension

**Objective**

Extend the portfolio ledger to support more complex financial transactions and stronger accounting verification.

**Tasks**

1. Support additional trade batches.
2. Track cumulative realized P&L.
3. Track total commissions.
4. Implement position-level realized P&L attribution.
5. Introduce configurable position limits.
6. Reject transactions violating portfolio constraints.
7. Maintain unique transaction identifiers.
8. Generate immutable ledger snapshots.
9. Support deterministic replay from a transaction history.
10. Verify accounting identities after every successful batch.
11. Test failed-batch rollback.
12. Compare results across transaction sequences.

**Requirements**

- Preserve long-only portfolio constraints.
- Use explicit financial units.
- Reject invalid numerical inputs.
- Avoid partially committed transaction batches.
- Implement numerical tolerance policies.
- Document limitations of average-cost accounting.

---

### Exercise 7: Concurrent State Transition Analysis

**Objective**

Investigate how concurrency affects assignment-driven financial state updates.

**Tasks**

1. Create a shared mutable portfolio ledger.
2. Simulate concurrent transaction submissions.
3. Identify possible consistency risks.
4. Introduce an explicit synchronization policy.
5. Protect related state updates.
6. Verify that public state readers observe consistent snapshots.
7. Test duplicate transaction submissions.
8. Test concurrent purchases competing for limited cash.
9. Compare synchronized and unsynchronized designs.
10. Document the limits of thread-based synchronization.

**Requirements**

- Use Python's standard library.
- Do not rely on the GIL as a transaction guarantee.
- Use deterministic concurrency tests where feasible.
- Avoid assuming a particular thread scheduling order.
- Explain the difference between thread safety, atomicity, and durability.



## Advanced Challenge: Auditable Transactional Portfolio State Engine

### Objective

Develop a production-oriented Python architecture for managing financial state transitions with controlled assignment semantics, validated operations, transaction replay, and comprehensive auditability.

The framework should distinguish between language-level assignment behavior and higher-level financial transaction guarantees.

### System Components

**1. Assignment Semantics Analyzer**

- AST-based assignment inspection.
- Name-target classification.
- Attribute-target classification.
- Subscription-target classification.
- Overlapping-target detection.
- Assignment complexity reporting.

**2. Financial Data Models**

- Immutable trade records.
- Immutable position records.
- Structured transaction results.
- Explicit financial units.
- Validated configuration objects.

**3. Transaction Validation**

- Numerical validation.
- Duplicate identifier detection.
- Cash sufficiency checks.
- Share availability checks.
- Position limit enforcement.
- Invalid-state detection.

**4. Staged Transaction Processing**

- Independent temporary state.
- Sequential transaction evaluation.
- Complete batch validation.
- Controlled state publication.
- Failure-state preservation.
- Version management.

**5. Accounting Engine**

- Average acquisition cost.
- Cash-flow tracking.
- Realized P&L.
- Transaction commissions.
- Position-level attribution.
- Accounting invariant verification.

**6. Concurrency Management**

- Explicit state synchronization.
- Controlled public state access.
- Consistent snapshot construction.
- Concurrent update testing.
- Documentation of synchronization guarantees.

**7. Audit and Replay**

- Transaction identifiers.
- Transaction sequence numbers.
- State versioning.
- Structured audit events.
- Deterministic replay.
- State consistency verification.

**8. Numerical Reliability**

- Explicit precision conventions.
- Floating-point error analysis.
- Decimal arithmetic evaluation.
- Reconciliation tests.
- Boundary-condition testing.

**9. Automated Quality Assurance**

- Unit testing.
- Regression testing.
- Failed-transaction tests.
- Concurrency tests.
- Numerical invariant tests.
- Static code analysis.
- Type checking.
- Continuous Integration.

### Advanced Engineering Requirements

1. Python 3.10 or later.
2. Modular Python architecture.
3. Explicit type annotations.
4. Controlled state mutation.
5. Immutable public snapshots.
6. No undocumented global-state dependencies.
7. Complete validation before commit.
8. Deterministic transaction replay.
9. Explicit synchronization.
10. Defined numerical precision policies.
11. Automated accounting verification.
12. Structured technical documentation.

### Expected Deliverables

1. Assignment semantics analysis module.
2. Financial transaction data models.
3. Portfolio ledger implementation.
4. Batch validation engine.
5. Transaction staging mechanism.
6. Controlled commit procedure.
7. Immutable snapshot functionality.
8. Transaction audit log.
9. Deterministic replay system.
10. Accounting reconciliation report.
11. Automated test suite.
12. Reproducible project configuration.
13. Technical architecture documentation.
14. Discussion of limitations and production requirements.

### Evaluation Criteria

| Dimension | Weight |
|---|---:|
| Assignment and state-transition correctness | 25% |
| Financial accounting accuracy | 20% |
| Transaction consistency | 20% |
| Testing and verification | 15% |
| Software architecture | 10% |
| Documentation and auditability | 10% |

### Final Goal

Develop a financial state-management framework that demonstrates precise understanding of Python assignment semantics while implementing stronger application-level consistency guarantees.

The framework should preserve valid financial state, reject invalid transitions, support reproducible transaction processing, and expose clearly documented limitations.
